# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. My rule and its reason codes

**The rule, in plain words:** *A page is worth a refresh review if it hasn't been touched in a
while AND it still gets meaningful search visibility.* Stale content that nobody sees isn't
urgent; stale content that's still pulling impressions is quietly decaying in public.

- **Reason code (one, fixed):** `stale_but_visible`
- **Action label:** `refresh_review`

Before I trust that rule, I check the two beliefs it leans on, each with a bucket table and n.
Both are signals behind real FlyRank flags from the session:

- **Signal 1 — staleness (`days_since_last_update`)** is the signal behind the *refresh* flag.
  Belief: "the longer since an update, the more likely a page is declining."
- **Signal 2 — CTR vs. position (`ctr` grouped by `position_tier`)** is the signal behind the
  *CTR-fix* logic. Belief: "CTR can't be judged on its own — it has to be judged against pages
  at a similar position, because position changes what a 'normal' CTR looks like."

Population for both checks: the full 30k-row starter slice (pseudonymized, one row per
content item). No future-window or label-derived column is used as an *input* anywhere below —
`trend_direction` is used **only** as the outcome I'm checking a belief against in Signal 1,
never as something the rule reads.


In [1]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"rows={len(df)}, cols={df.shape[1]}")

# Outcome used ONLY for validating beliefs below — never fed into the rule itself.
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# ---------------------------------------------------------------
# Signal 1 (flag-linked -> refresh flag): staleness vs decline rate
# ---------------------------------------------------------------
tier_order = ["0-30", "31-90", "91-180", "181+"]
signal1 = (
    df.groupby("freshness_tier")["is_declining"]
      .agg(n="count", decline_rate="mean")
      .reindex(tier_order)
)
signal1["decline_rate"] = signal1["decline_rate"].round(3)
print("\nSignal 1 — freshness_tier vs. decline rate")
print(signal1)

# Verdict logic: does decline rate climb monotonically with staleness, at a real sample size (n>=50)?
vals = signal1["decline_rate"].dropna()
monotonic = all(vals.iloc[i] <= vals.iloc[i + 1] for i in range(len(vals) - 1))
enough_n = (signal1["n"].dropna() >= 50).all()
signal1_verdict = "CONFIRMED" if (monotonic and enough_n) else "MIXED"
n_181plus = int(signal1.loc["181+", "n"])
print(f"\nSignal 1 verdict: {signal1_verdict} "
      f"(rate rises 0-30 -> 91-180, then drops at 181+, n={n_181plus} there — "
      f"staleness alone does not keep predicting decline)")

# ---------------------------------------------------------------
# Signal 2 (flag-linked -> CTR-fix flag): CTR vs. position tier
# ---------------------------------------------------------------
pos_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
signal2 = (
    df[df["position_tier"] != "no_data"]
      .groupby("position_tier")["ctr"]
      .agg(n="count", mean_ctr="mean", median_ctr="median")
      .reindex(pos_order)
)
signal2[["mean_ctr", "median_ctr"]] = signal2[["mean_ctr", "median_ctr"]].round(3)
print("\nSignal 2 — position_tier vs. CTR")
print(signal2)

ctr_vals = signal2["mean_ctr"]
ctr_monotonic_down = all(ctr_vals.iloc[i] >= ctr_vals.iloc[i + 1] for i in range(len(ctr_vals) - 1))
signal2_verdict = "CONFIRMED" if ctr_monotonic_down else "MIXED"
print(f"\nSignal 2 verdict: {signal2_verdict} "
      f"(mean CTR falls smoothly from top_3 to deep — a single global CTR threshold would be "
      f"unfair; CTR must be read against position peers, exactly like the CTR-fix flag does)")


rows=30000, cols=44

Signal 1 — freshness_tier vs. decline rate
                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611
181+              174         0.471

Signal 1 verdict: MIXED (rate rises 0-30 -> 91-180, then drops at 181+, n=174 there — staleness alone does not keep predicting decline)

Signal 2 — position_tier vs. CTR
                   n  mean_ctr  median_ctr
position_tier                             
top_3           2321     1.484        0.00
page_1         11814     0.652        0.16
striking        7304     0.323        0.11
page_3_5        7242     0.222        0.03
deep            1319     0.150        0.00

Signal 2 verdict: CONFIRMED (mean CTR falls smoothly from top_3 to deep — a single global CTR threshold would be unfair; CTR must be read against position peers, exactly like the CTR-fix flag does)


## 2. Build the ranked queue (writes the CSV)

Now I encode **one** rule the way the session built one live — four parts:

- **Population:** pages with real ranking data (`avg_position > 0`) and a real impression
  reading (`impression_tier != 'no_data'`) — no point flagging a page we have no signal on.
- **Evidence:** `days_since_last_update` (staleness) and `impressions_90d` (visibility).
- **Condition:** stale = `days_since_last_update >= 90` (this dataset's real "old" cluster —
  see the freshness_tier boundary above); visible = `impressions_90d >= 500`.
- **Action:** flagged pages get `reason_codes = "stale_but_visible"` and
  `suggested_action = "refresh_review"`; everything else gets `score = 0` and `action = "monitor"`.

The score itself is intentionally simple — no fitted weights, readable on purpose:
`score = stale_flag * visible_flag * impressions_90d`. Among flagged pages this naturally ranks
the *most-seen* stale pages first, which is exactly where a refresh pays off most.


In [2]:
STALE_DAYS = 90
VISIBLE_IMPRESSIONS = 500

eligible = (df["impression_tier"] != "no_data") & (df["avg_position"] > 0)
stale = df["days_since_last_update"] >= STALE_DAYS
visible = df["impressions_90d"] >= VISIBLE_IMPRESSIONS

flagged = eligible & stale & visible

out = df.copy()
out["score"] = 0.0
out.loc[flagged, "score"] = out.loc[flagged, "impressions_90d"].astype(float)

out["reason_codes"] = "none"
out.loc[flagged, "reason_codes"] = "stale_but_visible"

out["suggested_action"] = "monitor"
out.loc[flagged, "suggested_action"] = "refresh_review"

out["rank"] = out["score"].rank(method="first", ascending=False).astype(int)
out = out.sort_values("rank")

print(f"eligible population: {int(eligible.sum())}")
print(f"flagged (stale_but_visible): {int(flagged.sum())}")
print(f"share of eligible flagged: {flagged.sum() / eligible.sum():.3f}")

queue_cols = [
    "rank", "content_id", "client_id", "score", "reason_codes", "suggested_action",
    "impressions_90d", "clicks_90d", "ctr", "avg_position", "position_tier",
    "days_since_last_update", "freshness_tier", "content_type", "main_intent",
    "trend_direction",  # kept for human context in the CSV only — NOT used above as an input
]
queue = out[queue_cols]

out_path = Path("work/outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
queue.to_csv(out_path, index=False)
print(f"\nWrote {out_path} ({len(queue)} rows)")
queue.head(3)


eligible population: 28795
flagged (stale_but_visible): 6575
share of eligible flagged: 0.228



Wrote work/outputs/baseline_action_score.csv (30000 rows)


,rank,content_id,client_id,score,reason_codes,suggested_action,impressions_90d,clicks_90d,ctr,avg_position,position_tier,days_since_last_update,freshness_tier,content_type,main_intent,trend_direction
6653,1,content_5fe46e04994d,client_4e07408562,517715.0,stale_but_visible,refresh_review,517715,741,0.14,4.2,page_1,104,91-180,keyword article,informational,down
29400,2,content_2dba2b1f9536,client_6208ef0f77,443434.0,stale_but_visible,refresh_review,443434,910,0.21,27.9,page_3_5,104,91-180,keyword article,informational,stable
13537,3,content_2c2606c5d176,client_19581e27de,347399.0,stale_but_visible,refresh_review,347399,1854,0.53,4.2,page_1,104,91-180,keyword article,commercial,down


## 3. Top-10 review

Reading the actual top of the list — one line each: the action, why it's there, and what would
make this particular pick wrong. This is the human sanity check the session insisted on before
any list goes out.


In [3]:
top10 = queue.head(10).reset_index(drop=True)
pd.set_option("display.max_colwidth", 60)
top10_display = top10[[
    "rank", "content_id", "suggested_action", "reason_codes",
    "impressions_90d", "days_since_last_update", "avg_position", "ctr",
    "content_type", "trend_direction",
]]
top10_display


,rank,content_id,suggested_action,reason_codes,impressions_90d,days_since_last_update,avg_position,ctr,content_type,trend_direction
0,1,content_5fe46e04994d,refresh_review,stale_but_visible,517715,104,4.2,0.14,keyword article,down
1,2,content_2dba2b1f9536,refresh_review,stale_but_visible,443434,104,27.9,0.21,keyword article,stable
2,3,content_2c2606c5d176,refresh_review,stale_but_visible,347399,104,4.2,0.53,keyword article,down
3,4,content_cb112fce36be,refresh_review,stale_but_visible,309910,104,5.6,0.16,keyword article,down
4,5,content_9532f197bbc8,refresh_review,stale_but_visible,309192,104,2.0,0.87,keyword article,down
5,6,content_36ff89c8214e,refresh_review,stale_but_visible,295097,104,7.3,0.05,keyword article,stable
6,7,content_b28d1efd668f,refresh_review,stale_but_visible,286608,104,26.2,0.06,keyword article,stable
7,8,content_813e88069237,refresh_review,stale_but_visible,233561,104,26.2,0.06,keyword article,down
8,9,content_c21024970297,refresh_review,stale_but_visible,211366,104,5.1,0.41,keyword article,stable
9,10,content_c8e9d6ab9013,refresh_review,stale_but_visible,208678,104,9.7,0.00,keyword article,down


In [4]:
review_lines = []
for _, row in top10.iterrows():
    days = int(row["days_since_last_update"])
    impr = int(row["impressions_90d"])
    pos = row["avg_position"]
    why = f"flagged stale_but_visible: {days} days since update and {impr:,} impressions/90d (avg_position {pos:.1f})"
    if row["trend_direction"] in ("up", "stable"):
        trend = row["trend_direction"]
        wrong = f"would be wrong if: this page is already recovering on its own (trend_direction={trend}) — a refresh here may be wasted effort"
    elif row["content_type"] == "feedly article":
        wrong = "would be wrong if: this is syndicated/aggregated content the team doesn't actually own or edit"
    elif row["main_intent"] == "navigational":
        wrong = "would be wrong if: this page's low CTR is structural (branded/navigational query), not a content problem"
    else:
        wrong = "would be wrong if: the staleness is a client-side publishing gap (page frozen for a business reason), not neglect"
    review_lines.append({
        "rank": int(row["rank"]),
        "content_id": row["content_id"],
        "action": row["suggested_action"],
        "why_flagged": why,
        "what_would_make_it_wrong": wrong,
    })

review_df = pd.DataFrame(review_lines)
for _, r in review_df.iterrows():
    print(f"#{r['rank']:>2} {r['content_id']} -> {r['action']}")
    print(f"    why: {r['why_flagged']}")
    print(f"    what would make it wrong: {r['what_would_make_it_wrong']}")
    print()


# 1 content_5fe46e04994d -> refresh_review
    why: flagged stale_but_visible: 104 days since update and 517,715 impressions/90d (avg_position 4.2)
    what would make it wrong: would be wrong if: the staleness is a client-side publishing gap (page frozen for a business reason), not neglect

# 2 content_2dba2b1f9536 -> refresh_review
    why: flagged stale_but_visible: 104 days since update and 443,434 impressions/90d (avg_position 27.9)
    what would make it wrong: would be wrong if: this page is already recovering on its own (trend_direction=stable) — a refresh here may be wasted effort

# 3 content_2c2606c5d176 -> refresh_review
    why: flagged stale_but_visible: 104 days since update and 347,399 impressions/90d (avg_position 4.2)
    what would make it wrong: would be wrong if: the staleness is a client-side publishing gap (page frozen for a business reason), not neglect

# 4 content_cb112fce36be -> refresh_review
    why: flagged stale_but_visible: 104 days since update and 309,

## 4. Weak picks + leakage check

**Weak picks, read skeptically:** looking at the top 10 above, any row where
`trend_direction` is already `up` or `stable` is a weak pick — the rule flagged it purely on
staleness + visibility, but the page may already be recovering without a refresh, so a manual
refresh there is lower-value than it looks. That's exactly the risk Signal 1's MIXED verdict
predicted: staleness alone doesn't reliably track decline, so a few false positives in the top
10 are expected, not a bug in the code.

**Leakage check:** the rule's only inputs are `days_since_last_update`, `impressions_90d`,
`impression_tier`, and `avg_position` — all knowable *before* any decision is made about this
page, and none of them are derived from `trend_direction` / `trend_pct` (the label source) or
from any product-decision flag (this dataset has none, but the discipline applies regardless).
`trend_direction` appears in the output CSV only as a human-readable column for review — it
never entered the score, the condition, or the ranking.


In [5]:
weak = top10[top10["trend_direction"].isin(["up", "stable"])]
print(f"weak picks in top 10 (already up/stable despite being flagged): {len(weak)}")
print(weak[["rank", "content_id", "trend_direction", "days_since_last_update", "impressions_90d"]])

rule_inputs = {"days_since_last_update", "impressions_90d", "impression_tier", "avg_position"}
forbidden = {"trend_direction", "trend_pct"}
print(f"\nRule inputs used: {sorted(rule_inputs)}")
touched = rule_inputs & forbidden
print(f"Forbidden (label-derived) columns touched by the rule: {touched if touched else 'none'}")

metrics = {
    "signal_1_freshness_vs_decline": {
        "verdict": signal1_verdict,
        "buckets": signal1.reset_index().to_dict(orient="records"),
    },
    "signal_2_ctr_vs_position": {
        "verdict": signal2_verdict,
        "buckets": signal2.reset_index().to_dict(orient="records"),
    },
    "rule": {
        "population": "impression_tier != no_data AND avg_position > 0",
        "condition": f"days_since_last_update >= {STALE_DAYS} AND impressions_90d >= {VISIBLE_IMPRESSIONS}",
        "reason_code": "stale_but_visible",
        "action": "refresh_review",
        "eligible_population": int(eligible.sum()),
        "flagged_count": int(flagged.sum()),
    },
    "top_10_weak_picks": int(len(weak)),
}
metrics_path = Path("work/outputs/w04_baseline_metrics.json")
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"\nWrote {metrics_path}")


weak picks in top 10 (already up/stable despite being flagged): 4
   rank            content_id trend_direction  days_since_last_update  \
1     2  content_2dba2b1f9536          stable                     104   
5     6  content_36ff89c8214e          stable                     104   
6     7  content_b28d1efd668f          stable                     104   
8     9  content_c21024970297          stable                     104   

   impressions_90d  
1           443434  
5           295097  
6           286608  
8           211366  

Rule inputs used: ['avg_position', 'days_since_last_update', 'impression_tier', 'impressions_90d']
Forbidden (label-derived) columns touched by the rule: none

Wrote work/outputs/w04_baseline_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
